In [ ]:
!wget https://gist.githubusercontent.com/caiohamamuraIFSP/9b1677014666e4d17db6dd8c2e9c2bf6/raw/bd213b9b46096aab22b72eb7335d815919a40c49/machado2.txt

--2025-11-15 01:23:49--  https://gist.githubusercontent.com/caiohamamuraIFSP/9b1677014666e4d17db6dd8c2e9c2bf6/raw/bd213b9b46096aab22b72eb7335d815919a40c49/machado2.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1046375 (1022K) [text/plain]
Saving to: ‘machado2.txt.5’

machado2.txt.5      100%[===================>]   1022K  --.-KB/s    in 0.004s  

2025-11-15 01:23:49 (225 MB/s) - ‘machado2.txt.5’ saved [1046375/1046375]



In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
seq_len = 64
batch_size = 64
num_batches = 10000
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('machado2.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("pierreguillou/gpt2-small-portuguese")

vocab_size = tok.vocab_size

encode = lambda s: tok.encode(s, add_special_tokens=False)
decode = lambda ids: tok.decode(ids)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
vocab_size

50257

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------------
# helper constant
# ----------------------------------
LOG_10000 = torch.log(torch.tensor(10000.0).to(torch.bfloat16).cuda())


# ----------------------------------
# Positional Encoding
# ----------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)

        position = torch.arange(max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, embed_dim, 2).float() * (-LOG_10000 / embed_dim))

        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)

        pe = pe.unsqueeze(0)               # (1, max_len, embed_dim)
        self.register_buffer("pe", pe)      # moves with model.to(device)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]


# ----------------------------------
# Transformer Block
# ----------------------------------
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_mult=4, dropout=0.1):
        super().__init__()

        self.mha = nn.MultiheadAttention(
            embed_dim,
            num_heads=num_heads,
            batch_first=True,
            dropout=dropout
        )

        self.ln1 = nn.LayerNorm(embed_dim)
        self.dropout1 = nn.Dropout(dropout)

        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_hidden_mult * embed_dim),
            nn.ReLU(),
            nn.Linear(ff_hidden_mult * embed_dim, embed_dim),
        )

        self.ln2 = nn.LayerNorm(embed_dim)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, key_padding_mask=None):

        # Self attention
        attn, _ = self.mha(
            x, x, x,
            attn_mask=attn_mask,               # (seq, seq)
            key_padding_mask=key_padding_mask  # (batch, seq)
        )

        x = self.ln1(x + self.dropout1(attn))

        # Feedforward
        ff = self.ff(x)
        x = self.ln2(x + self.dropout2(ff))

        return x


# ----------------------------------
# Proper causal mask
# ----------------------------------
def causal_mask(seq_len):
    # boolean mask (True = block)
    # shape: (seq, seq)
    mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
    return mask


# ----------------------------------
# Full model
# ----------------------------------
class SimpleMHA(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, num_heads=4, n_layers=3, max_len=2048):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos = PositionalEncoding(embed_dim, max_len)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_hidden_mult=4, dropout=0.1)
            for _ in range(n_layers)
        ])

        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        """
        x: (batch, seq)
        returns logits for last token: (batch, vocab)
        """

        out = self.embed(x)
        out = self.pos(out)

        seq_len = out.size(1)
        mask = causal_mask(seq_len).to(out.device)

        for block in self.blocks:
            out = block(out, attn_mask=mask)

        out = out[:, -1, :]       # only final position for autoregressive task
        return self.fc(out)


In [ ]:
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long).to('cuda')
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [ ]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - seq_len, (batch_size,))
    x = torch.stack([data[i:i+seq_len] for i in ix])
    y = torch.stack([data[i+1:i+seq_len+1] for i in ix])
    return x, y

In [ ]:
# -------------------------------
# Training loop
# -------------------------------
embed_dim = 1024  # d_model
model = SimpleMHA(vocab_size, num_heads=16, n_layers=24, embed_dim=embed_dim).to(torch.bfloat16).to('cuda')
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)  # lr is ignored



In [ ]:
# Count number of parameters of model
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 405285969


In [ ]:
best_loss = float('inf')
best_model_weights = None
patience = 2000
cur_patience = 0

In [32]:

for step in range(num_batches):

    cur_patience += 1
    if cur_patience >= patience:
        print("Early stopping due to no improvement.")
        break

    # --------------------------
    # Training step
    # --------------------------
    model.train()
    x, y = get_batch('train')

    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        logits = model(x)
        logits = logits.view(-1, vocab_size)

        y = y[:, -1].view(-1)  # training only last token
        loss = F.cross_entropy(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # --------------------------
    # Validation step
    # --------------------------
    model.eval()
    with torch.no_grad():
        x_val, y_val = get_batch('validate')
        logits_val = model(x_val).view(-1, vocab_size)
        y_val = y_val[:, -1].view(-1)
        loss_val = F.cross_entropy(logits_val, y_val)

    if step % 20 == 0:
        print(f"step {step} loss {loss.item():.4f}  val {loss_val.item():.4f}")

    # Save best
    if loss_val.item() < best_loss:
        best_loss = loss_val.item()
        best_model_weights = model.state_dict()
        cur_patience = 0

/tmp/ipython-input-2538321745.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):


step 0 loss 6.6030  val 6.6562
step 20 loss 6.5361  val 6.9688
step 40 loss 7.6550  val 7.0312
step 60 loss 6.7620  val 6.7812
step 80 loss 7.4236  val 7.3438
step 100 loss 7.1064  val 6.3438
step 120 loss 5.9631  val 6.9688
step 140 loss 6.4927  val 7.1250
step 160 loss 6.5676  val 7.5000
step 180 loss 6.1943  val 6.9688
step 200 loss 7.0369  val 7.1250
step 220 loss 7.0520  val 6.8125
step 240 loss 7.2129  val 7.4688
step 260 loss 7.4495  val 7.0625
step 280 loss 6.8164  val 6.7500
step 300 loss 7.2896  val 6.6250
step 320 loss 6.7539  val 6.7188
step 340 loss 7.0811  val 6.3438
step 360 loss 7.1121  val 6.8438
step 380 loss 6.7412  val 7.4688
step 400 loss 6.1475  val 6.5625
step 420 loss 7.6609  val 7.1250
step 440 loss 6.6482  val 6.7188
step 460 loss 7.1980  val 6.7188
step 480 loss 6.5459  val 7.0312
step 500 loss 6.3450  val 7.2500
step 520 loss 7.1631  val 6.9688
step 540 loss 6.9170  val 7.7812
step 560 loss 6.9978  val 7.1250
step 580 loss 6.6189  val 6.8750
step 600 loss 6.

In [33]:
# Load best weights at the end
model.load_state_dict(best_model_weights)

print(f'Best loss: {best_loss}')

Best loss: 5.5


In [34]:
# Losses from best model
x, y = get_batch('train')
model.eval()
logits = model(x)
logits = logits.view(-1, vocab_size)
y = y[:, -1].view(-1)
loss = F.cross_entropy(logits, y)
print(f'Train loss: {loss.item():.4f} ({best_loss:.4f})')

Train loss: 6.5938 (5.5000)


In [35]:
top_k = 30
initial_text = 'O que'
full_context = torch.tensor(encode(initial_text), dtype=torch.long).to('cuda')
full_context = full_context.view(1, len(full_context))
#full_context = torch.zeros((1, 1), dtype=torch.long, device='cuda')
context = full_context

for _ in range(500):
    # Cut down context if larger than seq_len
    if context.size(1) > seq_len:
        context = context[:, -seq_len:]
    logits = model(context)
    # Temperature
    T = 0.7
    # Top-k filtering
    probs = F.softmax(logits/T, dim=-1)
    topk_probs, topk_indices = torch.topk(probs, top_k, dim=-1)  # (B, k)
    probs_zeroed = torch.zeros_like(probs).scatter_(-1, topk_indices, topk_probs)
    probs = probs_zeroed / probs_zeroed.sum(dim=-1, keepdim=True)
    next_id = torch.multinomial(probs, num_samples=1)
    context = torch.cat((context, next_id), dim=1)
    full_context = torch.cat((full_context, next_id), dim=1)

print(decode(full_context[0].tolist()))

O que um,, de que.— de o.
.,.?, e. não mas.
,., à e-, que. de um, a se..,,, se e. que. de de; a e, o. o-. não.. o-, a—-. e de,,.-,. a os.
,,-., e
— o os do por
-, se não.se para de,.... para de o, por,.;, não,,..,, o a e, o. o.
.
 a., não do um.., e o,. que de
.,.., os e..,; com.- a não. a., é..,- de de. e um que,
.. a-, do.,.,.—,.,, de..,—, é,. os. os o, de, da da,-
. para de. o era-, se,.., um a, do
 o, que
 que, que da e um,
.., com não.- a-? de, e,,,.— a,,,,-.,,,,
 que
. o.., uma de de.., de,. do, de,, o não. da e se, por para,,, que e., que. um,, que que,, que, se— a que do e,.. a, que. e,
,.,. a, o;,;, a não— que, de a por.,.,,,,
.
. a o-.
 não o que, de de que—,,., em,
.
 que.,. que,, é,. a, que,. e e. por que-. que

.— e,,.. as da,,, e não, se de e com de. a,.,
,.- da não e que,,
 que a. que,.,-,., de em e é.,
,, e,.- o se a.,,, se, do de;. uma que; e- o e não


In [ ]:
# Generate 500 tokens random text
rand = torch.randint(0, vocab_size, (500,))
print(decode(rand.tolist()))